In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages


# ============================================================
# SETTINGS
# ============================================================

ROOT = Path(".").resolve()

FEATURE_SETS = [30, 20, 12]


# ============================================================
# COLORS
# ============================================================

# Other supervised models
BACKGROUND_COLOR = "#BDBDBD"

# Selected model
RAW_COLOR = "#E67E22"          # without calibration
CALIBRATED_COLOR = "#7B4AB3"   # with calibration


# ============================================================
# LOAD CHI DATA
# ============================================================

def load_chi_data(root, transition):

    folder = root / "SUPERVISED" / transition

    chi_path = folder / f"{transition}_curve_chi.csv"

    if not chi_path.exists():
        raise FileNotFoundError(chi_path)

    chi_df = pd.read_csv(chi_path)

    return chi_df


# ============================================================
# FILTER CHI CURVE
# ============================================================

def filter_chi_curve(
    df,
    feature_set,
    model,
    x_column,
    calibration=None,
):
    """
    Select one model, feature set and optionally calibration.

    calibration:
        None             -> both calibration types
        "uncalibrated"   -> without calibration
        "calibrated"     -> with calibration
    """

    result = df[
        (df["feature_set"] == feature_set)
        & (df["model"] == model)
    ].copy()

    if calibration is not None:

        if "calibration" not in result.columns:
            raise ValueError(
                "Column 'calibration' not found in chi data."
            )

        result = result[
            result["calibration"] == calibration
        ].copy()

    return result.sort_values(x_column)


# ============================================================
# GET SUPERVISED MODELS
# ============================================================

def get_supervised_models(df):

    return sorted(
        df["model"]
        .dropna()
        .unique()
    )


# ============================================================
# CHECK CALIBRATION VALUES
# ============================================================

def check_calibration_values(df):

    if "calibration" not in df.columns:
        raise ValueError(
            "Column 'calibration' not found in chi data."
        )

    values = (
        df["calibration"]
        .dropna()
        .unique()
    )

    print("Available calibration values:")
    print(values)
    print()

    return values


# ============================================================
# PLOT OTHER SUPERVISED MODELS — GRAY
# ============================================================

def plot_chi_background(
    ax,
    chi_df,
    feature_set,
    selected_model,
    x_column,
):
    """
    Plot all supervised models except the selected one
    in gray.

    Both calibration versions are included.
    """

    models = (
        chi_df["model"]
        .dropna()
        .unique()
    )

    for other_model in models:

        # Do not plot the model currently being compared
        if other_model == selected_model:
            continue

        curve = filter_chi_curve(
            df=chi_df,
            feature_set=feature_set,
            model=other_model,
            x_column=x_column,
            calibration=None,
        )

        if curve.empty:
            continue

        ax.errorbar(
            curve[x_column],
            curve["chi"],
            yerr=curve["chi_err"],
            fmt="o-",
            color=BACKGROUND_COLOR,
            ecolor=BACKGROUND_COLOR,
            markersize=2,
            linewidth=0.9,
            capsize=1.5,
            elinewidth=0.6,
            alpha=0.40,
            zorder=1,
        )


# ============================================================
# PLOT ONE CHI CALIBRATION PANEL
# ============================================================

def plot_chi_calibration_panel(
    ax,
    chi_df,
    feature_set,
    model,
    x_column,
):
    """
    One panel:

        gray       = all other supervised models
        orange     = selected model, uncalibrated
        purple     = selected model, calibrated
    """

    # ========================================================
    # OTHER SUPERVISED MODELS
    # ========================================================

    plot_chi_background(
        ax=ax,
        chi_df=chi_df,
        feature_set=feature_set,
        selected_model=model,
        x_column=x_column,
    )

    # ========================================================
    # SELECTED MODEL — WITHOUT CALIBRATION
    # ========================================================

    uncalibrated_curve = filter_chi_curve(
        df=chi_df,
        feature_set=feature_set,
        model=model,
        x_column=x_column,
        calibration="uncalibrated",
    )

    if not uncalibrated_curve.empty:

        ax.errorbar(
            uncalibrated_curve[x_column],
            uncalibrated_curve["chi"],
            yerr=uncalibrated_curve["chi_err"],
            fmt="o-",
            color=RAW_COLOR,
            ecolor=RAW_COLOR,
            markersize=3.5,
            linewidth=2.0,
            capsize=2.5,
            elinewidth=0.9,
            label="bez kalibracji",
            zorder=4,
        )

    # ========================================================
    # SELECTED MODEL — WITH CALIBRATION
    # ========================================================

    calibrated_curve = filter_chi_curve(
        df=chi_df,
        feature_set=feature_set,
        model=model,
        x_column=x_column,
        calibration="calibrated",
    )

    if not calibrated_curve.empty:

        ax.errorbar(
            calibrated_curve[x_column],
            calibrated_curve["chi"],
            yerr=calibrated_curve["chi_err"],
            fmt="o-",
            color=CALIBRATED_COLOR,
            ecolor=CALIBRATED_COLOR,
            markersize=3.5,
            linewidth=2.0,
            capsize=2.5,
            elinewidth=0.9,
            label="z kalibracją",
            zorder=5,
        )

    # ========================================================
    # PANEL TITLE
    # ========================================================

    ax.set_title(
        f"{feature_set} cech",
        fontsize=13,
        fontweight="bold",
        pad=10,
    )

    # ========================================================
    # X AXIS
    # ========================================================

    ax.set_xlabel(
        r"$K_0$" if x_column == "K0" else r"$\Delta$",
        fontsize=11,
    )

    # ========================================================
    # Y AXIS
    # ========================================================

    ax.set_ylabel(
        r"$\chi$",
        fontsize=11,
    )

    # ========================================================
    # GRID
    # ========================================================

    ax.grid(
        alpha=0.25,
        linewidth=0.6,
    )

    ax.tick_params(
        axis="both",
        labelsize=9,
    )


# ============================================================
# CREATE ONE FIGURE FOR ONE MODEL
# ============================================================

def make_chi_calibration_figure(
    transition,
    model,
    chi_df,
    x_column,
):
    """
    Create one figure for one supervised model.

    Three panels:
        30 features
        20 features
        12 features
    """

    fig, axes = plt.subplots(
        nrows=1,
        ncols=3,
        figsize=(13.5, 4.5),
        sharey=True,
    )

    # ========================================================
    # PANELS
    # ========================================================

    for ax, feature_set in zip(
        axes,
        FEATURE_SETS,
    ):

        plot_chi_calibration_panel(
            ax=ax,
            chi_df=chi_df,
            feature_set=feature_set,
            model=model,
            x_column=x_column,
        )

    # ========================================================
    # Y LABEL
    # ========================================================

    axes[0].set_ylabel(
        r"$\chi$",
        fontsize=11,
    )

    # ========================================================
    # LEGEND
    # ========================================================

    handles, labels = axes[0].get_legend_handles_labels()

    if handles:

        fig.legend(
            handles,
            labels,
            loc="lower center",
            bbox_to_anchor=(0.5, -0.025),
            ncol=2,
            frameon=False,
            fontsize=9,
        )

    # ========================================================
    # TITLE
    # ========================================================

    fig.suptitle(
        f"{transition} — {model} — $\\chi$",
        fontsize=14,
        fontweight="bold",
        y=1.02,
    )

    # ========================================================
    # LAYOUT
    # ========================================================

    plt.tight_layout(
        rect=[
            0.0,
            0.08,
            1.0,
            0.96,
        ]
    )

    return fig, axes


# ============================================================
# CREATE ALL FIGURES FOR ONE TRANSITION
# ============================================================

def make_all_chi_calibration_figures(
    transition,
    pdf,
):
    """
    Create one figure for every supervised model
    for one transition.
    """

    # ========================================================
    # X AXIS
    # ========================================================

    if transition == "AC":

        x_column = "K0"

    elif transition == "BCb" or "AB":

        x_column = "Delta"

    else:

        raise ValueError(
            f"Unknown transition: {transition}"
        )

    # ========================================================
    # LOAD DATA
    # ========================================================

    chi_df = load_chi_data(
        ROOT,
        transition,
    )

    # ========================================================
    # CHECK CALIBRATION
    # ========================================================

    check_calibration_values(
        chi_df
    )

    # ========================================================
    # GET MODELS
    # ========================================================

    models = get_supervised_models(
        chi_df
    )

    print("=" * 70)
    print(f"TRANSITION: {transition}")
    print("=" * 70)
    print("Supervised models:")
    print(models)
    print()

    # ========================================================
    # CREATE ONE FIGURE PER MODEL
    # ========================================================

    for model in models:

        print(
            f"Plotting: {transition} — {model}"
        )

        fig, axes = make_chi_calibration_figure(
            transition=transition,
            model=model,
            chi_df=chi_df,
            x_column=x_column,
        )

        # Add figure to the same PDF
        pdf.savefig(
            fig,
            bbox_inches="tight",
        )

        plt.close(fig)


# ============================================================
# RUN
# ============================================================

output_dir = (
    ROOT
    / "figures_calibration_comparison"
)

output_dir.mkdir(
    parents=True,
    exist_ok=True,
)

pdf_path = (
    output_dir
    / "supervised_calibration_chi_comparison.pdf"
)


with PdfPages(pdf_path) as pdf:

    # --------------------------------------------------------
    # AC
    # --------------------------------------------------------

    make_all_chi_calibration_figures(
        "AC",
        pdf,
    )

    # --------------------------------------------------------
    # BCb
    # --------------------------------------------------------

    make_all_chi_calibration_figures(
        "BCb",
        pdf,
    )
    make_all_chi_calibration_figures(
        "AB",
        pdf,
    )


print(
    f"Saved: {pdf_path}"
)

Available calibration values:
['calibrated' 'uncalibrated']

TRANSITION: AC
Supervised models:
['Decision Tree', 'Gradient Boosted Trees', 'Logistic Regression', 'Logistic Regression C = 0.00175', 'Neural Network', 'Random Forest', 'SVM (RBF)', 'kNN', 'kNN ball_tree', 'kNN kd_tree']

Plotting: AC — Decision Tree
Plotting: AC — Gradient Boosted Trees
Plotting: AC — Logistic Regression
Plotting: AC — Logistic Regression C = 0.00175
Plotting: AC — Neural Network
Plotting: AC — Random Forest
Plotting: AC — SVM (RBF)
Plotting: AC — kNN
Plotting: AC — kNN ball_tree
Plotting: AC — kNN kd_tree
Available calibration values:
['calibrated' 'uncalibrated']

TRANSITION: BCb
Supervised models:
['Decision Tree', 'Gradient Boosted Trees', 'Logistic Regression', 'Logistic Regression C = 0.00175', 'Neural Network', 'Random Forest', 'SVM (RBF)', 'kNN', 'kNN ball_tree', 'kNN kd_tree']

Plotting: BCb — Decision Tree
Plotting: BCb — Gradient Boosted Trees
Plotting: BCb — Logistic Regression
Plotting: BCb — 